# Imports

In [1]:
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
import importlib

import datetime
import json
import os

import pandas as pd

import tagbiopy.utils
import tagbiopy.plots
import tagbiopy.regression.elastic_net

In [ ]:
importlib.reload(tagbiopy.regression.elastic_net)

In [ ]:
importlib.reload(tagbiopy.utils)

# Project

## FC json

In [3]:
fc_packet = './80894670699932446722590808882586350613.json'

## User function

In [4]:
py_script = '/Users/damir/code/protocols/customers/pfizer/fc-topaz/python/elastic_net_cross_validation.py'

In [5]:
user_function = tagbiopy.utils.load_function(py_script)

In [ ]:
user_function

# Download expression dataframe

In [ ]:
tag_data = tagbiopy.utils.TagbioData(fc_packet)
tag_data.entity_id = 'patient_id'

In [ ]:
tag_data

In [ ]:
extension = 'pdf'
fig_path = 'test_' + tagbiopy.utils.now().replace(' ', '-') + '.' + extension
tag_result = tagbiopy.utils.TagbioResult(extension=extension, path=fig_path, path_mutable=False)
tag_result.fh = open('from_notebook_{}.log'.format(tagbiopy.utils.now().replace(' ', '-')), 'w')

In [ ]:
%%time

tag_result = user_function(tag_data, tag_result)

In [ ]:
tag_result.fig

In [ ]:
tag_result.save()

# Plos One data

In [ ]:
project_name = 'topaz'
customer = 'pfizer'

DATA_BASE_DIR = '/Users/damir/data/{}-data'.format(project_name)
SOURCE_DATA_DIR = os.path.join(DATA_BASE_DIR, 'source_data')
PROCESSED_DATA_DIR = os.path.join(DATA_BASE_DIR, 'processed')
MODELS_DATA = os.path.join(DATA_BASE_DIR, 'models')

FEATURES_FILE = os.path.join(PROCESSED_DATA_DIR, 'snyder_et_al_processed_data-features.tsv')
FEATURE_CLASSES_FILE = os.path.join(PROCESSED_DATA_DIR, 'snyder_et_al_processed_data-feature-classes.tsv')
OUTCOMES_FILE = os.path.join(PROCESSED_DATA_DIR, 'snyder_et_al_processed_data-outcomes.tsv')

FEATURE_CLASSES = pd.read_csv(FEATURE_CLASSES_FILE, sep='\t').Class.value_counts().index.to_list()

In [ ]:
X = pd.read_csv(FEATURES_FILE, index_col=0, sep='\t')
y = pd.read_csv(OUTCOMES_FILE, index_col=0, sep='\t', squeeze=True)
feature_classes = pd.read_csv(FEATURE_CLASSES_FILE, index_col=0, sep='\t')

outcome_name = y.name
patients = X.index

In [ ]:
X.shape

In [ ]:
%%time
y_hat_tag, tag_model_params = \
    tagbiopy.regression.elastic_net.elastic_net_cross_validation(y, X)

In [ ]:
fig_tb = tagbiopy.plots.r2_plot(y, y_hat_tag, title='tagbiopy.regression.elastic_net: Plos One data')

In [ ]:
fig_tb

# Check differences b/w Plos One and FC data

In [ ]:
plos_one_to_tag_columns = {
    'Age': 'Model input: Age', 
    'Albumin < 4': 'Model input: Albumin < 4', 
    'Baseline neutrophil to lymphocyte ratio': 'Model input: Baseline neutrophil to lymphocyte ratio',
    'Clonality': 'Model input: Clonality: PBMC, Time point A', 
    'Clonality_tumor': 'Model input: Clonality: Tumor, Time point A', 
    'Diversity': 'Model input: Diversity: PBMC, Time point A',
    'Diversity_tumor': 'Model input: Diversity: Tumor, Time point A',
    'Number of chemo regimens total': 'Model input: Number of chemo regimens total',
    'Prior BCG': 'Model input: Prior BCG',
    'Productive Unique TCRs (cnt)': 'Model input: Productive Unique TCRs (cnt): PBMC, Time point A',
    'T-cell fraction': 'Model input: T-cell fraction: PBMC, Time point A',
    'T-cell fraction_tumor': 'Model input: T-cell fraction: Tumor, Time point A', 
    'Time since last chemotherapy': 'Model input: Time since last chemotherapy',
    'Top Clone Freq(%)': 'Model input: Top Clone Freq(%): PBMC, Time point A', 
    'expressed_missense_snv_count': 'Model input: expressed_missense_snv_count',
    'expressed_neoantigen_count': 'Model input: expressed_neoantigen_count', 
    'factor score': 'Model input: 5-factor score', 
    'log_Age': 'Model input: log Age',
    'log_Baseline neutrophil to lymphocyte ratio': 'Model input: log Baseline neutrophil to lymphocyte ratio', 
    'log_Clonality': 'Model input: log Clonality: PBMC, Time point A',
    'log_Clonality_tumor': 'Model input: log Clonality: Tumor, Time point A', 
    'log_Diversity': 'Model input: log Diversity: PBMC, Time point A', 
    'log_Diversity_tumor': 'Model input: log Diversity: Tumor, Time point A',
    'log_Number of chemo regimens total': 'Model input: log Number of chemo regimens total',
    'log_Productive Unique TCRs (cnt)': 'Model input: log Productive Unique TCRs (cnt): PBMC, Time point A', 
    'log_T-cell fraction': 'Model input: log T-cell fraction: PBMC, Time point A',
    'log_T-cell fraction_tumor': 'Model input: log T-cell fraction: Tumor, Time point A', 
    'log_Time since last chemotherapy': 'Model input: log Time since last chemotherapy',
    'log_Top Clone Freq(%)': 'Model input: log Top Clone Freq(%): PBMC, Time point A', 
    'log_expressed_missense_snv_count': 'Model input: log expressed_missense_snv_count',
    'log_expressed_neoantigen_count': 'Model input: log expressed_neoantigen_count', 
    'log_factor score': 'Model input: log 5-factor score',
    'log_missense_snv_count': 'Model input: log missense_snv_count', 
    'log_neoantigen_count': 'Model input: log neoantigen_count', 
    'missense_snv_count': 'Model input: missense_snv_count',
    'neoantigen_count': 'Model input: neoantigen_count'
}

In [ ]:
def concat_columns(s1, s2):
    return pd.concat([s1, s2], axis=1)

In [ ]:
_data = []

#c = outcome_name
#tc = tag_result.df.columns[0]
#w = concat_columns(y, tag_result.df[tc])
#delta = w.astype('float64').diff(axis=1).iloc[:,-1].sum()
#line = line = [c, tc, delta]
#_data.append(line)

for c in X.columns:
    tc = plos_one_to_tag_columns[c]
    w = concat_columns(X[c], tag_result.df[tc])
    delta = w.astype('float64').diff(axis=1).iloc[:,-1].sum()
    
    mean_plos_one = X[c].mean()
    mean_tb = tag_result.df[tc].mean()
    
    nans_plos_one = X[c].isna().sum()
    nan_tag_bio = tag_result.df[tc].isna().sum()

    line = [c, tc, nans_plos_one, nan_tag_bio, delta, mean_plos_one, mean_tb]
    _data.append(line)
    
diff_df = pd.DataFrame(data=_data, columns=['Plos One', 'Tag.bio',  'NaN Plos One', 'NaN Tag.bio',  'Difference', 'Mean Plos One', 'Mean Tag.bio'])

In [ ]:
diff_df